# Olist 통합 전처리: Distribution Hub 분석용

이 노트북은 기존 `HY/전처리.ipynb`의 집계 및 QA 흐름에 `SG/dataset_preprocessing.ipynb`의 구매시점/지역 파생변수를 통합합니다.

## 데이터셋 기준 단위 (grain)
- 최종 데이터셋의 한 행은 **주문 상품 1건** (`order_id` + `order_item_id`)입니다.
- 결제와 리뷰는 주문별로 집계한 뒤 조인하여 1:N 조인으로 인한 행 증식을 방지합니다.
- 주문 상태는 제거하지 않고 유지하며, 배송 분석용 `delivered` 데이터셋을 별도로 생성합니다.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

SEARCH_ROOTS = [Path.cwd(), *Path.cwd().parents]
DATA_DIR = next(
    (root / "data" for root in SEARCH_ROOTS if (root / "data" / "olist_orders_dataset.csv").exists()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("프로젝트 루트의 data 폴더를 찾을 수 없습니다.")

OUTPUT_DIR = DATA_DIR / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("input :", DATA_DIR.resolve())
print("output:", OUTPUT_DIR.resolve())

## 1. 원본 CSV 로드 및 기본 점검

In [ ]:
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
geolocation = pd.read_csv(DATA_DIR / "olist_geolocation_dataset.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
order_payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")

raw_tables = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation,
}
pd.DataFrame({name: {"rows": len(df), "columns": df.shape[1]} for name, df in raw_tables.items()}).T

In [ ]:
def audit_table(df, name, key_cols=()):
    result = {"table": name, "rows": len(df), "columns": df.shape[1], "null_cells": int(df.isna().sum().sum())}
    for key in key_cols:
        result[f"duplicate_{key}"] = int(df[key].duplicated().sum())
    return result

audits = [
    audit_table(orders, "orders", ["order_id"]),
    audit_table(customers, "customers", ["customer_id"]),
    audit_table(order_items, "order_items", []),
    audit_table(products, "products", ["product_id"]),
    audit_table(sellers, "sellers", ["seller_id"]),
]
display(pd.DataFrame(audits))
display(orders["order_status"].value_counts(dropna=False).rename("orders"))

## 2. 주문 전처리 및 배송 파생변수

취소 주문을 처음부터 삭제하지 않습니다. 목적에 맞는 필터를 후속 분석 데이터셋에서 명시적으로 적용합니다.

In [ ]:
orders_clean = orders.copy()
order_dt_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in order_dt_cols:
    orders_clean[col] = pd.to_datetime(orders_clean[col], errors="coerce")

purchase_ts = orders_clean["order_purchase_timestamp"]
orders_clean["purchase_date"] = purchase_ts.dt.normalize()
orders_clean["purchase_year"] = purchase_ts.dt.year.astype("Int64")
orders_clean["purchase_month"] = purchase_ts.dt.month.astype("Int64")
orders_clean["purchase_weekday"] = purchase_ts.dt.day_name()
orders_clean["purchase_hour"] = purchase_ts.dt.hour.astype("Int64")

orders_clean["anomaly_carrier_before_approved"] = (
    orders_clean["order_delivered_carrier_date"] < orders_clean["order_approved_at"]
)
orders_clean["anomaly_customer_before_carrier"] = (
    orders_clean["order_delivered_customer_date"] < orders_clean["order_delivered_carrier_date"]
)
required_delivery_date_cols = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
orders_clean["anomaly_missing_required_delivery_date"] = (
    orders_clean[required_delivery_date_cols].isna().any(axis=1)
)
orders_clean["anomaly_flag"] = (
    orders_clean["anomaly_carrier_before_approved"]
    | orders_clean["anomaly_customer_before_carrier"]
    | orders_clean["anomaly_missing_required_delivery_date"]
).astype("int8")

orders_clean["delivery_days_all"] = (
    orders_clean["order_delivered_customer_date"] - orders_clean["order_purchase_timestamp"]
).dt.days.astype("Int64")
orders_clean["delivery_days_clean"] = orders_clean["delivery_days_all"].where(orders_clean["anomaly_flag"] == 0)
late_missing = orders_clean["order_delivered_customer_date"].isna() | orders_clean["order_estimated_delivery_date"].isna()
orders_clean["is_late"] = (
    orders_clean["order_delivered_customer_date"] > orders_clean["order_estimated_delivery_date"]
).astype("boolean").mask(late_missing)
orders_clean["is_delivered"] = orders_clean["order_status"].eq("delivered")

display(orders_clean[["delivery_days_all", "delivery_days_clean"]].describe())
display(orders_clean[[
    "anomaly_carrier_before_approved",
    "anomaly_customer_before_carrier",
    "anomaly_missing_required_delivery_date",
]].sum().rename("orders_by_anomaly_reason"))
display(orders_clean["anomaly_flag"].value_counts().rename("orders"))

## 3. 고객/판매자 지역 및 상품 전처리

우편번호별 위치는 평균 위경도와 최빈 주(state)로 축약합니다. 판매자 원본 주와 우편번호 기반 주가 다른 경우에는 원본을 임의 수정하지 않고 검증 플래그로 남깁니다.

In [ ]:
STATE_KOR_MAP = {
    "AC": "아크리주", "AL": "알라고아스주", "AP": "아마파주", "AM": "아마조나스주",
    "BA": "바이아주", "CE": "세아라주", "DF": "연방구", "ES": "이스피리투산투주",
    "GO": "고이아스주", "MA": "마라냥주", "MT": "마투그로수주", "MS": "마투그로수두술주",
    "MG": "미나스제라이스주", "PA": "파라주", "PB": "파라이바주", "PR": "파라나주",
    "PE": "페르남부쿠주", "PI": "피아우이주", "RJ": "리우데자네이루주",
    "RN": "히우그란지두노르치주", "RS": "히우그란지두술주", "RO": "혼도니아주",
    "RR": "호라이마주", "SC": "산타카타리나주", "SP": "상파울루주",
    "SE": "세르지피주", "TO": "토칸칭스주",
}
REGION_MAP = {
    **{state: "북부" for state in ["AC", "AP", "AM", "PA", "RO", "RR", "TO"]},
    **{state: "북동부" for state in ["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"]},
    **{state: "중서부" for state in ["DF", "GO", "MT", "MS"]},
    **{state: "남동부" for state in ["ES", "MG", "RJ", "SP"]},
    **{state: "남부" for state in ["PR", "RS", "SC"]},
}

def first_mode(series):
    mode = series.dropna().mode()
    return mode.iloc[0] if not mode.empty else pd.NA

geo_zip = geolocation.groupby("geolocation_zip_code_prefix", as_index=False).agg(
    geo_lat=("geolocation_lat", "mean"),
    geo_lng=("geolocation_lng", "mean"),
    geo_city=("geolocation_city", first_mode),
    geo_state=("geolocation_state", first_mode),
)

customers_clean = customers.copy()
customers_clean["customer_state_kor"] = customers_clean["customer_state"].map(STATE_KOR_MAP)
customers_clean["customer_region"] = customers_clean["customer_state"].map(REGION_MAP)
customer_geo = geo_zip.add_prefix("customer_").rename(columns={"customer_geolocation_zip_code_prefix": "customer_zip_code_prefix"})
customers_clean = customers_clean.merge(customer_geo, on="customer_zip_code_prefix", how="left", validate="many_to_one")

sellers_clean = sellers.copy()
sellers_clean["seller_state_kor"] = sellers_clean["seller_state"].map(STATE_KOR_MAP)
sellers_clean["seller_region"] = sellers_clean["seller_state"].map(REGION_MAP)
seller_geo = geo_zip.add_prefix("seller_").rename(columns={"seller_geolocation_zip_code_prefix": "seller_zip_code_prefix"})
sellers_clean = sellers_clean.merge(seller_geo, on="seller_zip_code_prefix", how="left", validate="many_to_one")
sellers_clean["seller_state_geo_mismatch"] = (
    sellers_clean["seller_geo_state"].notna() & sellers_clean["seller_state"].ne(sellers_clean["seller_geo_state"])
)

products_clean = products.merge(category_translation, on="product_category_name", how="left", validate="many_to_one")
print("seller state/geolocation 불일치:", int(sellers_clean["seller_state_geo_mismatch"].sum()))
print("번역 누락 상품:", int(products_clean["product_category_name_english"].isna().sum()))

## 4. 다건 테이블의 주문 단위 집계

`payments`와 `reviews`를 먼저 주문 단위로 집계하므로 주문 상품 데이터와 결합해도 결제/리뷰 다건으로 행이 추가되지 않습니다.

In [ ]:
order_reviews_clean = order_reviews.copy()
for col in ["review_creation_date", "review_answer_timestamp"]:
    order_reviews_clean[col] = pd.to_datetime(order_reviews_clean[col], errors="coerce")

pay_agg = order_payments.groupby("order_id", as_index=False).agg(
    payment_value_total=("payment_value", "sum"),
    payment_installments_max=("payment_installments", "max"),
    payment_type_nunique=("payment_type", "nunique"),
    payment_types=("payment_type", lambda x: ", ".join(sorted(x.dropna().unique()))),
)
rev_agg = order_reviews_clean.groupby("order_id", as_index=False).agg(
    review_score_mean=("review_score", "mean"),
    review_count=("review_id", "count"),
    review_creation_min=("review_creation_date", "min"),
    review_answer_max=("review_answer_timestamp", "max"),
)

print("결제 원본/집계 행:", len(order_payments), "/", len(pay_agg))
print("리뷰 원본/집계 행:", len(order_reviews), "/", len(rev_agg))
assert not pay_agg["order_id"].duplicated().any()
assert not rev_agg["order_id"].duplicated().any()

## 5. 주문-상품 단위 최종 데이터셋 결합

In [ ]:
order_items_clean = order_items.copy()
order_items_clean["item_total"] = order_items_clean["price"] + order_items_clean["freight_value"]

olist_item_df = (
    order_items_clean
    .merge(orders_clean, on="order_id", how="left", validate="many_to_one")
    .merge(customers_clean, on="customer_id", how="left", validate="many_to_one")
    .merge(sellers_clean, on="seller_id", how="left", validate="many_to_one")
    .merge(products_clean, on="product_id", how="left", validate="many_to_one")
    .merge(pay_agg, on="order_id", how="left", validate="many_to_one")
    .merge(rev_agg, on="order_id", how="left", validate="many_to_one")
)
olist_item_df["seller_customer_same_state"] = olist_item_df["seller_state"].eq(olist_item_df["customer_state"])
olist_item_df["delivery_route"] = olist_item_df["seller_state"] + " -> " + olist_item_df["customer_state"]

delivered_item_df = olist_item_df.loc[
    olist_item_df["is_delivered"] & olist_item_df["anomaly_flag"].eq(0)
].copy()

print("전체 주문-상품 데이터:", olist_item_df.shape)
print("배송 분석용 데이터  :", delivered_item_df.shape)
display(olist_item_df.head())

## 6. 최종 QA

- 최종 행 수가 `order_items` 행 수와 동일한지 검증합니다.
- 주문별 상품 합계와 주문별 결제 합계의 차이를 점검합니다.
- 배송 분석 데이터는 `delivered`이면서 날짜 역전 또는 주요 배송 날짜 결측 이상치가 없는 건만 포함합니다.

In [ ]:
assert len(olist_item_df) == len(order_items), "조인 과정에서 주문-상품 행 수가 변했습니다."
assert not olist_item_df.duplicated(["order_id", "order_item_id"]).any(), "주문-상품 키가 중복되었습니다."

missing_summary = olist_item_df[[
    "customer_id", "seller_id", "product_id", "payment_value_total", "product_category_name_english"
]].isna().sum().rename("missing_rows")

item_sum = olist_item_df.groupby("order_id", as_index=False).agg(item_total=("item_total", "sum"))
amount_check = item_sum.merge(pay_agg[["order_id", "payment_value_total"]], on="order_id", how="left", validate="one_to_one")
amount_check["payment_minus_item"] = amount_check["payment_value_total"] - amount_check["item_total"]

print("조인 행 수 보존: OK -", len(olist_item_df), "rows")
display(missing_summary)
display(amount_check["payment_minus_item"].describe(percentiles=[0.01, 0.5, 0.99]))
print("금액 차이 절대값 > 1 주문 수:", int(amount_check["payment_minus_item"].abs().gt(1).sum()))
print("배송 분석 데이터 주문 상태:", delivered_item_df["order_status"].unique())

## 7. 처리 결과 저장

- `olist_order_item_level.csv`: 상태를 포함한 전체 주문-상품 데이터
- `olist_delivered_item_level.csv`: 배송 분석에 바로 사용할 수 있는 정상 배송 데이터

In [ ]:
all_output_path = OUTPUT_DIR / "olist_order_item_level.csv"
delivered_output_path = OUTPUT_DIR / "olist_delivered_item_level.csv"

olist_item_df.to_csv(all_output_path, index=False, encoding="utf-8-sig")
delivered_item_df.to_csv(delivered_output_path, index=False, encoding="utf-8-sig")

print("saved:", all_output_path.resolve(), olist_item_df.shape)
print("saved:", delivered_output_path.resolve(), delivered_item_df.shape)